# Adaptive RAG

Not every query needs your best model. "What year was the Eiffel Tower built?" and "Analyse the trade-offs between Germany's emissions trading scheme and Japan's carbon tax given the structure of each country's industrial sector" are both valid user queries, but they differ enormously in the reasoning depth required. Routing them to the same model wastes money on simple queries and under-serves complex ones. Adaptive RAG uses a lightweight complexity classifier to route each query: simple queries go to a fast, cheap model; complex queries go to a stronger, slower model. The result is lower median cost without sacrificing quality on the queries that need it.

**What you'll build:** A RAG pipeline that scores each query for complexity, routes it to `fast_llm` or `strong_llm` accordingly, and logs routing decisions so you can tune the threshold over time.

**Time:** ~20 min | **Difficulty:** Advanced

## Prerequisites

- Completed the [Basic RAG](https://synapsekit.github.io/synapsekit-docs/guides/rag/) guide
- Familiarity with prompt routing patterns
- `OPENAI_API_KEY` set in your environment

## What you'll learn

- How `AdaptiveRAGPipeline` classifies query complexity
- How to configure routing thresholds and model assignments
- How to inspect routing decisions and tune the threshold against your query log
- When complexity routing is worth the overhead and when it is not

In [ ]:
!pip install synapsekit -q

## Environment Setup

In [ ]:
import os

# os.environ["OPENAI_API_KEY"] = "sk-..."  # set your key here or via environment

## Step 1: Install and configure

In [ ]:
import asyncio
import os

from synapsekit.llms.openai import OpenAILLM
from synapsekit.embeddings.openai import OpenAIEmbeddings
from synapsekit.vectorstores.memory import InMemoryVectorStore
from synapsekit.pipelines import AdaptiveRAGPipeline

# Two models at different capability/cost trade-offs
fast_llm = OpenAILLM(model="gpt-4o-mini")   # cheap and fast for simple queries
strong_llm = OpenAILLM(model="gpt-4o")       # more capable for complex reasoning

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = InMemoryVectorStore(embeddings=embeddings)

## Step 2: Load documents

In [ ]:
docs = [
    "The Eiffel Tower was completed in 1889 and stands 330 metres tall.",
    "Paris is the capital of France with a population of approximately 2.1 million.",
    "Germany's Energiewende policy targets 80% renewable electricity by 2030.",
    "Germany's carbon pricing scheme covers industry and transport under an emissions trading system.",
    "Japan's Green Growth Strategy targets carbon neutrality by 2050 via hydrogen and ammonia.",
    "Japan introduced a carbon tax in 2012 at a rate of \u00a5289 per tonne of CO2.",
    "The EU Emissions Trading System (ETS) is a cap-and-trade scheme covering heavy industry.",
    "Carbon taxes set a fixed price per tonne; emissions trading caps total volume and lets the market set the price.",
    "Japan's automotive industry accounts for 10% of GDP and relies heavily on exports.",
    "Germany's automotive sector employs 800,000 workers and faces transition risk from the 2035 ICE ban.",
]

await vectorstore.aadd(docs)
print(f"Added {len(docs)} documents to the vector store.")

## Step 3: Configure AdaptiveRAGPipeline

The `complexity_threshold` is the single most important tuning parameter. Queries scoring above the threshold are routed to `strong_llm`. Start at 0.5 and adjust after reviewing routing logs on your actual query distribution.

In [ ]:
pipeline = AdaptiveRAGPipeline(
    fast_llm=fast_llm,
    strong_llm=strong_llm,
    vectorstore=vectorstore,
    embeddings=embeddings,
    complexity_threshold=0.5,   # queries above this go to strong_llm
    top_k=5,
)
print("AdaptiveRAGPipeline configured with complexity_threshold=0.5.")

## Step 4: Inspect complexity scoring before running full queries

In [ ]:
async def score_queries():
    queries = [
        "What year was the Eiffel Tower completed?",                          # simple lookup
        "What is the population of Paris?",                                    # simple lookup
        "How does Germany's emissions trading scheme work?",                   # moderate
        "Analyse the trade-offs between Germany's ETS and Japan's carbon tax "
        "and explain how each policy affects industrial competitiveness.",      # complex
    ]

    for query in queries:
        score = await pipeline.aclassify_complexity(query)
        route = "strong_llm" if score >= 0.5 else "fast_llm"
        print(f"[{score:.2f} -> {route}] {query[:70]}")

asyncio.run(score_queries())
# Expected:
# [0.08 -> fast_llm]   What year was the Eiffel Tower completed?
# [0.11 -> fast_llm]   What is the population of Paris?
# [0.41 -> fast_llm]   How does Germany's emissions trading scheme work?
# [0.83 -> strong_llm] Analyse the trade-offs between Germany's ETS and Japan's carbo...

## Step 5: Run queries through the adaptive pipeline

In [ ]:
async def adaptive_query(question: str):
    result = await pipeline.aquery(question)
    print(f"Q: {question}")
    print(f"   Routed to: {result.model_used} (complexity: {result.complexity_score:.2f})")
    print(f"A: {result.answer[:200]}\n")

asyncio.run(adaptive_query("What year was the Eiffel Tower completed?"))
asyncio.run(adaptive_query(
    "Analyse the trade-offs between Germany's ETS and Japan's carbon tax "
    "and explain how each policy affects industrial competitiveness."
))

## Step 6: Stream responses with routing metadata

In [ ]:
async def stream_with_routing():
    question = "Analyse the trade-offs between Germany's ETS and Japan's carbon tax."
    print(f"Q: {question}\n")
    async for chunk in pipeline.astream(question):
        if chunk.is_metadata:
            print(f"[routed to {chunk.model_used}, complexity={chunk.complexity_score:.2f}]\n")
        else:
            print(chunk.text, end="", flush=True)
    print()

asyncio.run(stream_with_routing())

## Step 7: Review routing statistics across a query batch

After running the pipeline on a sample of your real queries, review the routing distribution to check whether the threshold is set appropriately. If 90% of queries are routed to `strong_llm`, the threshold is too low.

In [ ]:
async def batch_routing_stats():
    test_queries = [
        "What year was the Eiffel Tower completed?",
        "What is the population of Paris?",
        "How does Germany's carbon pricing work?",
        "Compare Germany and Japan's climate policies.",
        "What is the EU ETS?",
        "Analyse how Japan's carbon tax affects automotive export competitiveness.",
        "How tall is the Eiffel Tower?",
        "What are the trade-offs between carbon taxes and cap-and-trade systems?",
    ]

    stats = await pipeline.arun_batch(test_queries, return_stats=True)
    print(f"Total queries: {stats.total}")
    print(f"Routed to fast_llm:   {stats.fast_count} ({stats.fast_pct:.0f}%)")
    print(f"Routed to strong_llm: {stats.strong_count} ({stats.strong_pct:.0f}%)")
    print(f"Estimated cost saving vs all-strong: {stats.cost_saving_pct:.0f}%")

asyncio.run(batch_routing_stats())

## Step 8: Tune the threshold using routing labels

Once you have accumulated real query routing data, you can evaluate whether the threshold is calibrated correctly by comparing model-used to human-labelled query complexity.

In [ ]:
# After accumulating logs, adjust the threshold and re-run
pipeline.complexity_threshold = 0.6  # raise threshold if strong_llm is used too often
print(f"Threshold updated to: {pipeline.complexity_threshold}")

# Verify with a quick check
async def check_threshold():
    score = await pipeline.aclassify_complexity("What year was the Eiffel Tower completed?")
    route = "strong_llm" if score >= pipeline.complexity_threshold else "fast_llm"
    print(f"Simple query complexity: {score:.2f} -> {route}")

asyncio.run(check_threshold())

## Complete Working Example

A single self-contained cell that runs the entire Adaptive RAG pipeline end-to-end.

In [ ]:
import asyncio
import os

from synapsekit.llms.openai import OpenAILLM
from synapsekit.embeddings.openai import OpenAIEmbeddings
from synapsekit.vectorstores.memory import InMemoryVectorStore
from synapsekit.pipelines import AdaptiveRAGPipeline

DOCS = [
    "The Eiffel Tower was completed in 1889 and stands 330 metres tall.",
    "Paris is the capital of France with a population of approximately 2.1 million.",
    "Germany's Energiewende policy targets 80% renewable electricity by 2030.",
    "Germany's carbon pricing scheme covers industry and transport under an emissions trading system.",
    "Japan's Green Growth Strategy targets carbon neutrality by 2050 via hydrogen and ammonia.",
    "Japan introduced a carbon tax in 2012 at a rate of \u00a5289 per tonne of CO2.",
    "The EU Emissions Trading System (ETS) is a cap-and-trade scheme covering heavy industry.",
    "Carbon taxes set a fixed price per tonne; emissions trading caps total volume and lets the market set the price.",
    "Japan's automotive industry accounts for 10% of GDP and relies heavily on exports.",
    "Germany's automotive sector employs 800,000 workers and faces transition risk from the 2035 ICE ban.",
]

QUERIES = [
    ("simple", "What year was the Eiffel Tower completed?"),
    ("simple", "What is the population of Paris?"),
    ("moderate", "How does Germany's emissions trading scheme work?"),
    ("complex", "Analyse the trade-offs between Germany's ETS and Japan's carbon tax "
                "and explain how each policy affects industrial competitiveness."),
]


async def main():
    fast_llm = OpenAILLM(model="gpt-4o-mini")
    strong_llm = OpenAILLM(model="gpt-4o")
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    vectorstore = InMemoryVectorStore(embeddings=embeddings)

    await vectorstore.aadd(DOCS)

    pipeline = AdaptiveRAGPipeline(
        fast_llm=fast_llm,
        strong_llm=strong_llm,
        vectorstore=vectorstore,
        embeddings=embeddings,
        complexity_threshold=0.5,
        top_k=5,
    )

    for expected_complexity, question in QUERIES:
        result = await pipeline.aquery(question)
        print(f"[{expected_complexity.upper()}] Q: {question[:70]}")
        print(f"   Complexity score: {result.complexity_score:.2f} -> {result.model_used}")
        print(f"   A: {result.answer[:180]}\n")


asyncio.run(main())

## Next Steps

- [Self-RAG](https://synapsekit.github.io/synapsekit-docs/guides/retrieval/self-rag) — add quality grading for queries routed to the strong model
- [RAG Fusion](https://synapsekit.github.io/synapsekit-docs/guides/retrieval/rag-fusion) — apply multi-query fusion only for complex-routed queries to further improve recall
- [Cross-Encoder Reranking](https://synapsekit.github.io/synapsekit-docs/guides/retrieval/cross-encoder-reranking) — use a reranker selectively on strong-routed queries to maximise precision where it matters most